# How sensitive are the candidates to screening settings?

Evaluate once, then compare three prespecified mirror ratios. Continuous scores and model predictions stay fixed. This is a parameter check, not a search for a preferred result.

In [ ]:
import pandas as pd
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
result = gauge.evaluate(
    adata,
    groupby="group",
    views={
        "molecular": {"obsm": "X_pca"},
        "qc": {
            "obs": ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
            "log1p": ["total_counts", "n_genes_by_counts"],
        },
    },
)

## Compare the same scores

Smaller ratios are stricter. The +1 term requires at least 5, 3 or 2 positive-tail groups at 0.2, 1/3 or 0.5, respectively—even with no negative-tail groups. Use `1 / 3`, not `0.33`.

In [ ]:
rows = []
for label, ratio in [("0.2", 0.2), ("1/3", 1 / 3), ("0.5", 0.5)]:
    audit = gauge.audit(
        result, reference="molecular", contrast="qc", maximum_mirror_ratio=ratio
    )
    info = audit.metadata["screening"]
    rows.append({
        "Maximum mirror ratio": label,
        "Required positives (minimum)": info["minimum_positive_count"],
        "Threshold": info["threshold"],
        "Positive tail": info["positive_tail_count"],
        "Negative tail": info["negative_tail_count"],
        "Achieved ratio": info["achieved_mirror_ratio"],
        "Candidates": ", ".join(audit.table.loc[audit.table.candidate, "group"]) or "None",
        "Reason": info["reason"],
    })
comparison = pd.DataFrame(rows).set_index("Maximum mirror ratio")
comparison.round(3).astype(object).where(comparison.notna(), "—")

Here, 0.2 cannot select candidates: only three groups have positive contrast. The 1/3 and 0.5 settings select Q0–Q2. Missing tail counts mean no threshold was selected, not that the observed tails are empty.

Agreement across settings describes parameter sensitivity; it does not establish biological truth or error-rate control. Keep the continuous contrasts and report the settings you checked. [Decision fields and interpretation](../docs/api.md#screening-explanations).

## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to inspect this saved evaluation and assess how descriptive screening changes with the mirror ratio and eligibility requirement. Choose a useful sensitivity range for the available group support, and report both stable contrasts and changes in candidate calls.

These tutorial settings are an example; adapt them to your question and data.